In [ ]:
import numpy as np
import nbtlib
from nbtlib.tag import Compound, ByteArray, Int, String, List
import json
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def create_schem_from_npy(npy_path, json_path, output_dir):
    """Создает .schem файл из .npy и .json файлов в формате SpongeV2."""
    try:
        structure = np.load(npy_path)
        if structure.ndim != 3:
            logging.error(f"Файл {npy_path} не является 3D массивом.")
            return False

        structure = np.transpose(structure, (2, 1, 0))
        logging.info(f"Порядок осей после транспонирования: (2, 1, 0) -> (x, z, y)")

        unique_values = np.unique(structure).astype(str)
        logging.info(f"Уникальные значения в .npy: {unique_values}")
        logging.info(f"Размер структуры: {structure.shape}, первый срез: {structure[0, 0, :]}")

        with open(json_path, "r") as f:
            block_library = json.load(f)
        
        for val in unique_values:
            if val not in block_library and val != "0":
                logging.warning(f"Значение {val} из .npy отсутствует в .json, будет заменено на minecraft:air")

        width, length, height = structure.shape  
        logging.info(f"Размеры структуры: Width={width}, Height={height}, Length={length}")

        schematic = Compound({
            "Version": Int(2),  
            "DataVersion": Int(2730),  
            "Width": Int(width),      
            "Height": Int(height),    
            "Length": Int(length),    
            "PaletteMax": Int(0),     
            "Palette": Compound(),
            "BlockData": ByteArray(),
            "BlockEntities": List(),
            "Entities": List()
        })

        palette = {}
        palette_index = 0

        palette["minecraft:air"] = Int(0)
        palette_index += 1

        for block_id in unique_values:
            if block_id == "0":  
                continue
            block_name = block_library.get(block_id, "minecraft:air")
            if block_name not in palette:
                palette[block_name] = Int(palette_index)
                logging.info(f"Добавлен блок в палитру: {block_name} -> индекс {palette_index}")
                palette_index += 1

        schematic["Palette"] = Compound(palette)
        schematic["PaletteMax"] = Int(palette_index)

        total_blocks = width * height * length
        block_data_bytes = bytearray(total_blocks)

        index = 0
        for y in range(height):    
            for z in range(length):  
                for x in range(width):  )
                    block_id = str(structure[x, z, y])
                    block_name = block_library.get(block_id, "minecraft:air")
                    block_index = palette.get(block_name, 0)  
                    block_data_bytes[index] = block_index
                    index += 1

        schematic["BlockData"] = ByteArray(block_data_bytes)
        logging.info(f"BlockData заполнен: {len(block_data_bytes)} байт")

        output_path = output_dir / f"{npy_path.stem}.schem"
        nbtlib.File(schematic).save(output_path, gzipped=True)
        logging.info(f"Создан файл {output_path}")
        return True

    except Exception as e:
        logging.error(f"Ошибка при обработке {npy_path}: {str(e)}")
        return False

def process_directory(input_dir, output_dir):
    """Обрабатывает все .npy файлы в директории и создает .schem файлы."""
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    npy_files = list(input_dir.rglob("*.npy"))
    if not npy_files:
        logging.warning(f"В директории {input_dir} не найдено .npy файлов.")
        return

    json_files = list(input_dir.rglob("*.json"))
    if not json_files:
        logging.error(f"В директории {input_dir} не найдено .json файлов.")
        return

    if len(json_files) == 1:
        json_path = json_files[0]
        logging.info(f"Используется общий JSON файл: {json_path}")
        for npy_path in npy_files:
            create_schem_from_npy(npy_path, json_path, output_dir)
    else:

        for npy_path in npy_files:
            json_path = npy_path.with_suffix(".json")
            if json_path.exists():
                logging.info(f"Обработка {npy_path} с JSON {json_path}")
                create_schem_from_npy(npy_path, json_path, output_dir)
            else:
                logging.warning(f"Для {npy_path} не найден соответствующий .json файл.")

if __name__ == "__main__":
    input_directory = "input_files"
    output_directory = "output_schem"
    process_directory(input_directory, output_directory)